In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import pad_sequences

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load reviews and inspect a small sample
df = pd.read_csv(dataset_root / "CBOW/IMDB Dataset.csv").dropna()
print("Dataset shape:", df.shape)
print("Sentiment counts:")
print(df.sentiment.value_counts())
print("Sample reviews:")
for i in range(3):
    print(i + 1, df.review.iloc[i][:200])


In [ ]:
# Clean reviews and encode their sentiment labels
sentences = []
for review in df.review:
    words = preprocess_string(review)
    sentences.append(words)
print("Number of documents:", len(sentences))
print("First review tokens:", sentences[0][:20])

encoder = LabelEncoder()
y = encoder.fit_transform(df.sentiment)
print("Sentiment encoding:")
for i in range(len(encoder.classes_)):
    print(encoder.classes_[i], "->", i)
print("First ten encoded labels:", y[:10])

train, test, y_train, y_test = train_test_split(sentences, y, test_size=0.2)
print("Training documents:", len(train))
print("Test documents:", len(test))


In [ ]:
# Train CBOW on training words before adding padding
cbow = Word2Vec(train, min_count=1, sg=0, epochs=20)
print("Vocabulary size:", len(cbow.wv))
print("Word vector size:", cbow.vector_size)
print("Sample vocabulary:", cbow.wv.index_to_key[:10])
sample_word = cbow.wv.index_to_key[0]
print("Embedding sample for", sample_word, ":", cbow.wv[sample_word][:10])


In [ ]:
# Pad or truncate each review to 300 tokens
train = pad_sequences(
    train, maxlen=300, dtype=object, value="", padding="post", truncating="post"
)
test = pad_sequences(
    test, maxlen=300, dtype=object, value="", padding="post", truncating="post"
)
print("Padded training shape:", train.shape)
print("Padded test shape:", test.shape)
print("First training review tokens:", train[0][:12])


In [ ]:
# Average word vectors for each review; empty padding is ignored
x_train = []
for words in train:
    vector = cbow.wv.get_mean_vector(words)
    x_train.append(vector)
x_train = np.array(x_train)

x_test = []
for words in test:
    vector = cbow.wv.get_mean_vector(words)
    x_test.append(vector)
x_test = np.array(x_test)
print("Training document vectors:", x_train.shape)
print("Test document vectors:", x_test.shape)
print("First document vector sample:", x_train[0][:10])


In [ ]:
# Build the sentiment classifier
model = Sequential([
    Input(shape=(cbow.vector_size,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
# Train the sentiment classifier
model.fit(x_train, y_train, epochs=20, batch_size=128)


In [ ]:
# Evaluate and inspect five predictions
loss, accuracy = model.evaluate(x_test, y_test)
print("Test loss:", loss)
print("Test accuracy:", accuracy * 100, "%")

probabilities = model.predict(x_test[:5], verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
results = pd.DataFrame({
    "True sentiment": encoder.inverse_transform(y_test[:5]),
    "Predicted sentiment": encoder.inverse_transform(predictions),
    "Positive probability": probabilities.round(4)
})
print(results.to_string(index=False))
